In [ ]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 "pandas<3" s3fs==2026.3.0 -q

# Import des packages nécessaires

In [ ]:
import pandas as pd
from openpyxl import load_workbook 
import os



# Import des données
Les données sont disponibles sur le site [DATA AMELI](https://data.ameli.fr/pages/home/).


In [ ]:
df_effectifs = pd.read_csv('../DATA/effectifs.csv', sep=';')
df_depenses = pd.read_csv('../DATA/depenses.csv', sep=';')
df_regions_dept = pd.read_csv('../DATA/regions_dept.csv', sep=';' ,encoding='latin1')
# Filtre sur 2020–2023
df_effectifs = df_effectifs[df_effectifs['annee'].between(2021, 2023)]
df_depenses = df_depenses[df_depenses['annee'].between(2021, 2023)]

# Harmonisation des colonnes les 1 devient des 01
df_regions_dept["Code département"] = df_regions_dept["Code département"].astype(str).str.zfill(2)
df_effectifs["dept"] = df_effectifs["dept"].astype(str).str.zfill(2)

df_final = pd.merge(
    df_effectifs, 
    df_regions_dept, 
    left_on="dept", 
    right_on="Code département", 
    how="inner"
)
#  La jointure entre le dataset effectifs et régions pour récuperer les libelleés
df_fusion = pd.merge(
    df_final,
    df_depenses,
    on=['top'],
    how='inner',
    suffixes=('_dept', '_national')
)


In [ ]:
df = df_fusion

In [ ]:
df.info()

# Les champs sont les suivants avant nettoyage:

In [ ]:
df.columns


# Nettoyage des données

In [ ]:
# Nettoyage des parenthèses
cols = ["patho_niv2", "patho_niv3"]
for c in cols:
    df_fusion.loc[:, c] = df_fusion[c].str.replace(r"\s*\(.*\)", "", regex=True)

# Remplacement des NaN de Ntop_dept par 0
df_fusion.loc[df_fusion['Ntop_dept'].isna(), 'Ntop_dept'] = 0

#  Nettoyage des NaN sur les pathologies uniquement
df_fusion = df_fusion.dropna(subset=['patho_niv3', 'patho_niv2','prev'])

# Grand filtre de nettoyage
liste_exclusions = ['999', '2A', '2B', '971', '972', '973', '974', '976']

df_fusion = df_fusion[
    # On garde uniquement les montants strictement positifs
    (df_fusion['montant'] > 0) & 
    
    # On exclut le 999, la Corse et tous les DOM
    (~df_fusion['Département'].astype(str).isin(liste_exclusions)) &
    
    # On enlève la région 99 ou 999 au cas où
    (df_fusion['Région'].astype(str) != '99')
]


In [ ]:
# Suppression des colonnes inutiles pour le nettoyage final
df = df_fusion.drop(
    columns=[
        "Code département",
        "tri_dept",
        "tri_national",
        "region",
        "dept",
        "Niveau prioritaire_dept",
        "Niveau prioritaire_national",
        "cla_age_5",
        "type_somme",
        "top",
        "sexe",
        "dep_niv_1",
        "patho_niv1"

    ],
    errors='ignore' # Évite de faire une erreur si ça a déjà été supprimée
)

In [ ]:
df

In [ ]:
df

# Valeurs manquantes

In [ ]:
(
    df
    .isna()
    .sum(axis=0)
)

In [ ]:
print(df['patho_niv2'].unique()) 
print(df['patho_niv3'].unique()) 



# Les champs sont les suivants après nettoyage:

In [ ]:
df.columns


In [ ]:
df

In [ ]:
df_fusion

In [ ]:
df.info()

# Création du fichier Excel

Je vais créer un nouveau fichier Excel qui contiendra les données nettoyées dans un onglet "cleaned_data".

In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.worksheet.array_formula import ArrayFormula

# Charger données
df = pd.read_csv('../DATA/effectifs.csv', sep=';')
file_path = '../DATA/Dashboard_filtres.xlsx'

# 1. Exporter données nettoyées
with pd.ExcelWriter(file_path, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='cleaned_data', index=False)

print("Données exportées")

# 2. Créer onglet Ressources
wb = load_workbook(file_path)

if 'Ressources' not in wb.sheetnames:
    ress = wb.create_sheet('Ressources', 0)
else:
    ress = wb['Ressources']

# Titres
ress['A1'] = 'Département'
ress['B1'] = 'Année'
ress['C1'] = 'Sexe'
ress['D1'] = 'Pathologie'

# Ajouter les valeurs UNIQUES
depts = sorted([str(x) for x in df['dept'].dropna().unique()])
annees = sorted(df['annee'].unique())
sexes = sorted(df['sexe'].unique())
pathos = sorted(df['patho_niv2'].dropna().unique())

# Département (colonne A)
for i, val in enumerate(depts, start=2):
    ress[f'A{i}'] = val

# Année (colonne B)
for i, val in enumerate(annees, start=2):
    ress[f'B{i}'] = val

# Sexe (colonne C)
for i, val in enumerate(sexes, start=2):
    ress[f'C{i}'] = val

# Pathologie (colonne D)
for i, val in enumerate(pathos, start=2):
    ress[f'D{i}'] = val

wb.save(file_path)
wb.close()



# DATA VIZ - EXCEL